## Download PJM RT-HRL LMP Historical Data

Downloads real-time hourly LMP data for **all PJM nodes** for periods older than the 731-day standard data window.
Historical data cannot be filtered by `pnode_id` at the API level — all nodes are downloaded and filtered afterwards.

- Test month: April 2024
- Stores results in a separate DuckDB: `pjm_lmp_historical.duckdb`

In [ ]:
import requests
import pandas as pd
import duckdb
import time
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}
BASE_URL = "https://api.pjm.com/api/v1/rt_hrl_lmps"

In [ ]:
# Load Virginia pnode IDs for post-download filtering
va_pnode_ids = pd.read_csv(DATA_DIR / "processed" / "usa_data" / "va_pnode_ids_final.csv")
va_pnode_id_set = set(va_pnode_ids["pnode_id"].tolist())
print(f"Virginia pnode IDs loaded: {len(va_pnode_id_set)}")

In [40]:
# Set up DuckDB
db_path = DATA_DIR / "processed" / "usa_data" / "pjm_lmp_historical.duckdb"
con = duckdb.connect(str(db_path))

con.execute("""
    CREATE TABLE IF NOT EXISTS rt_hrl_lmps (
        datetime_beginning_utc VARCHAR,
        datetime_beginning_ept VARCHAR,
        pnode_id BIGINT,
        pnode_name VARCHAR,
        voltage VARCHAR,
        equipment VARCHAR,
        type VARCHAR,
        zone VARCHAR,
        system_energy_price_rt DOUBLE,
        total_lmp_rt DOUBLE,
        congestion_price_rt DOUBLE,
        marginal_loss_price_rt DOUBLE,
        row_is_current BOOLEAN,
        version_nbr INTEGER
    )
""")

# Track completed months for resume capability
try:
    done_months = con.execute("""
        SELECT DISTINCT substr(datetime_beginning_ept, 1, 7) AS month
        FROM rt_hrl_lmps
    """).fetchdf()["month"].tolist()
    done_months_set = set(m for m in done_months if m)
except Exception:
    done_months_set = set()

print(f"Already completed months: {sorted(done_months_set)}")

Already completed months: ['2017-01', '2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07', '2018-08', '2018-09', '2018-10', '2018-11', '2018-12', '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12', '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06', '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12', '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04']


In [7]:
def fetch_historical_month(start_date, max_retries=5):
    """
    Download all PJM nodes for a full month (no pnode_id filter).
    Filters by type=LOAD;GEN;EHV to exclude AGGREGATE and other unused types.
    Paginates through all rows. Retries on 429 with backoff.
    Returns a DataFrame filtered to Virginia pnode_ids.
    """
    end_date = start_date + relativedelta(months=1) - relativedelta(days=1)
    date_str = (
        f"{start_date.month}-{start_date.day}-{start_date.year} 00:00 to "
        f"{end_date.month}-{end_date.day}-{end_date.year} 23:00"
    )

    all_rows = []
    start_row = 1
    page = 0

    while True:
        params = {
            "rowCount": 50000,
            "startRow": start_row,
            "datetime_beginning_ept": date_str,
            "row_is_current": 1,
            "type": "LOAD;GEN;EHV",
        }

        for attempt in range(max_retries):
            r = requests.get(BASE_URL, headers=headers, params=params)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = 10 * (2 ** attempt)  # 10s, 20s, 40s, 80s, 160s
                print(f"    429 rate limit — waiting {wait}s (attempt {attempt+1}/{max_retries})", flush=True)
                time.sleep(wait)
            else:
                print(f"    API error {r.status_code}: {r.text[:200]}")
                return pd.DataFrame(all_rows) if all_rows else pd.DataFrame()
        else:
            print("    Max retries exceeded, stopping pagination")
            break

        items = r.json().get("items", [])
        all_rows.extend(items)
        page += 1

        if page % 20 == 0:
            print(f"    Page {page} — {len(all_rows):,} rows so far", flush=True)

        if len(items) < 50000:
            break
        start_row += 50000
        time.sleep(10)  # 6 connections/min limit

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    # Filter to Virginia pnode_ids only
    if "pnode_id" in df.columns:
        df = df[df["pnode_id"].isin(va_pnode_id_set)]
    return df

In [8]:
# Test download: April 2024 only
test_date = datetime(2024, 4, 1)
month_key = test_date.strftime("%Y-%m")

if month_key in done_months_set:
    print(f"{month_key} already downloaded — skipping")
else:
    print(f"Downloading {month_key} (all PJM nodes, filter after)...", flush=True)
    t0 = time.time()

    df = fetch_historical_month(test_date)

    if not df.empty:
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        print(f"  Saved {len(df):,} Virginia rows for {month_key}")
    else:
        print(f"  No data returned for {month_key}")

    elapsed = time.time() - t0
    print(f"  Completed in {elapsed:.0f}s")

con.close()

    Page 20 — 1,000,000 rows so far
    Page 40 — 2,000,000 rows so far
    Page 60 — 3,000,000 rows so far
    Page 80 — 4,000,000 rows so far
    Page 100 — 5,000,000 rows so far
    Page 120 — 6,000,000 rows so far
    Page 140 — 7,000,000 rows so far
    Page 160 — 8,000,000 rows so far
    Page 180 — 9,000,000 rows so far
  Saved 779,760 Virginia rows for 2024-04
  Completed in 3873s


In [9]:
# Inspect results
con = duckdb.connect(str(db_path))
print("Total rows:", con.execute("SELECT COUNT(*) FROM rt_hrl_lmps").fetchone()[0])
print("\nDate range:")
print(con.execute("SELECT MIN(datetime_beginning_ept), MAX(datetime_beginning_ept) FROM rt_hrl_lmps").fetchone())
print("\nUnique pnode_ids:", con.execute("SELECT COUNT(DISTINCT pnode_id) FROM rt_hrl_lmps").fetchone()[0])
print("\nType breakdown:")
print(con.execute("SELECT type, COUNT(*) as n FROM rt_hrl_lmps GROUP BY type ORDER BY n DESC").fetchdf())
con.close()

Total rows: 779760

Date range:
('2024-04-01T00:00:00', '2024-04-30T23:00:00')

Unique pnode_ids: 1083

Type breakdown:
   type       n
0  LOAD  614160
1   GEN  139680
2   EHV   25920


In [ ]:
# Inspect a few rows
con = duckdb.connect(str(db_path))                                                                                                                                                         
con.execute("SELECT * FROM rt_hrl_lmps LIMIT 5").fetchdf()  

,datetime_beginning_utc,datetime_beginning_ept,pnode_id,pnode_name,voltage,equipment,type,zone,system_energy_price_rt,total_lmp_rt,congestion_price_rt,marginal_loss_price_rt,row_is_current,version_nbr
0,2024-04-01T04:00:00,2024-04-01T00:00:00,48907,GOSHEN,35 KV,BUS1,LOAD,PECO,15.03,15.75,0.54,0.18,True,1
1,2024-04-01T04:00:00,2024-04-01T00:00:00,48908,GOSHEN,35 KV,BUS2,LOAD,PECO,15.03,15.75,0.54,0.18,True,1
2,2024-04-01T04:00:00,2024-04-01T00:00:00,49283,ROCKRIDG,13 KV,LOAD3,LOAD,BGE,15.03,15.59,0.56,0.00,True,1
3,2024-04-01T04:00:00,2024-04-01T00:00:00,49284,ROCKRIDG,34 KV,LOAD1,LOAD,BGE,15.03,15.59,0.56,0.00,True,1
4,2024-04-01T04:00:00,2024-04-01T00:00:00,49285,ROCKRIDG,34 KV,LOAD2,LOAD,BGE,15.03,15.70,0.57,0.11,True,1


In [ ]:
#Check Zones overlap
hist_path = DATA_DIR / "processed" / "usa_data" / "pjm_lmp_historical.duckdb"                                                                                                              
filtered_path = DATA_DIR / "processed" / "usa_data" / "pjm_lmp_va_2yr.duckdb"
                                                                                                                                                                                            
con_hist = duckdb.connect(str(hist_path))                                                                                                                                                  
hist_zones = set(con_hist.execute("SELECT DISTINCT zone FROM rt_hrl_lmps").fetchdf()["zone"].tolist())                                                                                     
con_hist.close()                                                                                                                                                                           
                
con_filt = duckdb.connect(str(filtered_path))                                                                                                                                              
filt_zones = set(con_filt.execute("SELECT DISTINCT zone FROM rt_hrl_lmps").fetchdf()["zone"].tolist())
con_filt.close()                                                                                                                                                                           
                
print("In historical but not in filtered:", hist_zones - filt_zones)                                                                                                                       
print("In filtered but not in historical:", filt_zones - hist_zones)
print("Matching zones:", hist_zones & filt_zones)           

In historical but not in filtered: set()
In filtered but not in historical: set()
Matching zones: {'PECO', 'DUQ', 'PPL', 'METED', 'ATSI', 'DOM', 'PEPCO', 'APS', 'DAY', 'AEP', 'PSEG', 'EKPC', 'BGE', 'JCPL', 'DPL', 'PENELEC'}


In [16]:
# Download January – March 2024                                                                                                                                                            
start_date = datetime(2024, 1, 1)                                                                                                                                                          
end_date = datetime(2024, 3, 1)                                                                                                                                                            
                                                                                                                                                                                            
total_rows = 0
current = start_date                                                                                                                                                                       
                
con = duckdb.connect(str(db_path))

while current <= end_date:
    month_key = current.strftime("%Y-%m")

    if month_key in done_months_set:                                                                                                                                                       
        print(f"Skipping {month_key} (already downloaded)")
        current += relativedelta(months=1)                                                                                                                                                 
        continue

    print(f"Downloading {month_key}...", flush=True)
    t0 = time.time()
                                                                                                                                                                                            
    df = fetch_historical_month(current)
                                                                                                                                                                                            
    if not df.empty:
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        total_rows += len(df)

    elapsed = time.time() - t0                                                                                                                                                             
    print(f"  {month_key}: {len(df):,} rows in {elapsed:.0f}s — cumulative: {total_rows:,}", flush=True)
    current += relativedelta(months=1)                                                                                                                                                     
                
con.close()
print(f"\nDone. Total rows stored this run: {total_rows:,}")

    Page 20 — 1,000,000 rows so far
    Page 40 — 2,000,000 rows so far
    Page 60 — 3,000,000 rows so far
    Page 80 — 4,000,000 rows so far
    Page 100 — 5,000,000 rows so far
    Page 120 — 6,000,000 rows so far
    Page 140 — 7,000,000 rows so far
    Page 160 — 8,000,000 rows so far
    Page 180 — 9,000,000 rows so far
  2024-01: 804,264 rows in 4081s — cumulative: 804,264
    Page 20 — 1,000,000 rows so far
    Page 40 — 2,000,000 rows so far
    Page 60 — 3,000,000 rows so far
    Page 80 — 4,000,000 rows so far
    Page 100 — 5,000,000 rows so far
    Page 120 — 6,000,000 rows so far
    Page 140 — 7,000,000 rows so far
    Page 160 — 8,000,000 rows so far
    Page 180 — 9,000,000 rows so far
  2024-02: 752,376 rows in 3629s — cumulative: 1,556,640
    Page 20 — 1,000,000 rows so far
    Page 40 — 2,000,000 rows so far
    Page 60 — 3,000,000 rows so far
    Page 80 — 4,000,000 rows so far
    Page 100 — 5,000,000 rows so far
    Page 120 — 6,000,000 rows so far
    Page 140

In [17]:
con = duckdb.connect(str(db_path))                                                                                                                                                         
con.execute("SELECT DISTINCT equipment FROM rt_hrl_lmps ORDER BY equipment").fetchdf()   

,equipment
0,1 BANK
1,110-1
2,110-2
3,2 BANK
4,230-10LD
...,...
311,XT1
312,XT1L
313,XT2 LOAD
314,XT4LOAD


In [47]:
pd.set_option('display.max_rows', None)  

con = duckdb.connect(str(db_path))                                                                                                                                                         
print("Total rows:", con.execute("SELECT COUNT(*) FROM rt_hrl_lmps").fetchone()[0])                                                                                                        
print("\nRows per month:")                                                                                                                                                                 
print(con.execute("""                                                                                                                                                                      
    SELECT substr(datetime_beginning_ept, 1, 7) AS month, COUNT(*) as rows                                                                                                                 
    FROM rt_hrl_lmps                                                                                                                                                                       
    GROUP BY month
    ORDER BY month                                                                                                                                                                         
""").fetchdf()) 
print("\nUnique pnode_ids per month:")                                                                                                                                                     
print(con.execute("""                                                                                                                                                                      
    SELECT substr(datetime_beginning_ept, 1, 7) AS month, COUNT(DISTINCT pnode_id) as pnode_ids
    FROM rt_hrl_lmps                                                                                                                                                                       
    GROUP BY month
    ORDER BY month                                                                                                                                                                         
""").fetchdf())
con.close()  

Total rows: 57582896

Rows per month:
      month    rows
0   2017-01  690432
1   2018-01  712008
2   2018-02  643104
3   2018-03  714315
4   2018-04  694800
5   2018-05  719256
6   2018-06  699120
7   2018-07  722424
8   2018-08  722424
9   2018-09  701520
10  2018-10  729864
11  2018-11  706320
12  2018-12  731304
13  2019-01  732096
14  2019-02  661248
15  2019-03  733392
16  2019-04  712080
17  2019-05  737496
18  2019-06  717120
19  2019-07  741024
20  2019-08  741024
21  2019-09  718272
22  2019-10  746976
23  2019-11  722880
24  2019-12  748992
25  2020-01  749952
26  2020-02  701568
27  2020-03  750960
28  2020-04  728640
29  2020-05  753120
30  2020-06  730080
31  2020-07  754416
32  2020-08  754416
33  2020-09  733320
34  2020-10  761112
35  2020-11  736560
36  2020-12  761664
37  2021-01  761856
38  2021-02  688128
39  2021-03  761886
40  2021-04  738720
41  2021-05  765840
42  2021-06  744480
43  2021-07  769296
44  2021-08  769296
45  2021-09  745488
46  2021-10  771528
47

In [39]:
# Download all of 2023                                                                                                                                                                     
start_date = datetime(2017, 1, 1)                                                                                                                                                          
end_date = datetime(2017, 12, 1)                                                                                                                                                           
                                                                                                                                                                                            
total_rows = 0                                                                                                                                                                             
current = start_date                                                                                                                                                                       
                
con = duckdb.connect(str(db_path))                                                                                                                                                         

while current <= end_date:                                                                                                                                                                 
    month_key = current.strftime("%Y-%m")
                                                                                                                                                                                            
    if month_key in done_months_set:
        print(f"Skipping {month_key} (already downloaded)")                                                                                                                                
        current += relativedelta(months=1)                                                                                                                                                 
        continue
                                                                                                                                                                                            
    print(f"Downloading {month_key}...", flush=True)                                                                                                                                       
    t0 = time.time()
                                                                                                                                                                                            
    df = fetch_historical_month(current)                                                                                                                                                   

    if not df.empty:                                                                                                                                                                       
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        total_rows += len(df)                                                                                                                                                              

    elapsed = time.time() - t0                                                                                                                                                             
    print(f"  {month_key}: {len(df):,} rows in {elapsed:.0f}s — cumulative: {total_rows:,}", flush=True)
    current += relativedelta(months=1)                                                                                                                                                     
                                                                                                                                                                                            
con.close()                                                                                                                                                                                
print(f"\nDone. Total rows stored this run: {total_rows:,}")  

KeyboardInterrupt: 

In [42]:
#Check for duplicates
con = duckdb.connect(str(db_path))
con.execute("""
    SELECT COUNT(*) as duplicate_pairs      
    FROM (                                  
        SELECT datetime_beginning_ept, pnode_id
        FROM rt_hrl_lmps                                                                                                                                                                   
        GROUP BY datetime_beginning_ept, pnode_id                                                                                                                                          
        HAVING COUNT(*) > 1                                                                                                                                                                
    )                                                                                                                                                                                      
""").fetchone() 

(759304,)

In [46]:
#May 2025 downloads were duplicated, remove
#First back up the DB
con = duckdb.connect(str(db_path))                                                                                                                                                         
con.execute("""                                                                                                                                                                            
    COPY rt_hrl_lmps TO 'pjm_lmp_historical_backup.parquet' (FORMAT PARQUET)                                                                                                               
""")                                                                                                                                                                                       
print("Backup complete") 

# Remove duplicates                                                                                                                                                                        
con.execute(""" 
    DELETE FROM rt_hrl_lmps                                                                                                                                                                
    WHERE rowid NOT IN (
        SELECT MIN(rowid)                                                                                                                                                                  
        FROM rt_hrl_lmps                    
        GROUP BY datetime_beginning_ept, pnode_id
    )
""")                                                                                                                                                                                       
print("Rows after dedup:", con.execute("SELECT COUNT(*) FROM rt_hrl_lmps").fetchone()[0])
con.close() 

Backup complete
Rows after dedup: 57582896
